## Transformer 작성하기

### 문제 설명
PyTorch에서 필요한 섹션을 완성하여 **Transformer 모델**을 구현합니다. 이 모델은 임베딩 레이어, Transformer 인코더, 그리고 시퀀스 처리와 예측을 위한 출력 레이어로 구성됩니다.

### 요구사항
1. **Transformer 모델 구조 정의**:
   - **임베딩 레이어**:
     - 입력 데이터를 더 높은 차원의 공간으로 변환하는 레이어를 구현합니다.
     - 입력으로부터 임베딩을 만들기 위해 `torch.nn.Linear` 또는 `torch.nn.Embedding` 레이어를 사용합니다.
   - **Transformer 인코더**:
     - 어텐션을 사용해 시퀀스를 처리하기 위해 `torch.nn.TransformerEncoder` 또는 `torch.nn.Transformer`를 사용합니다.
     - 어텐션 헤드 수와 인코더 레이어 수 같은 파라미터를 설정합니다.
   - **출력 레이어**:
     - Transformer의 시퀀스 출력을 원하는 출력 차원으로 줄이기 위해 완전 연결(선형) 레이어를 추가합니다.

2. **forward 메서드 구현**:
   - 임베딩 레이어를 사용해 입력을 더 높은 차원 공간으로 매핑합니다.
   - 변환된 입력을 Transformer 인코더에 통과시킵니다.
   - 출력 레이어를 사용해 인코딩된 시퀀스를 예측값으로 변환합니다.

### 제약 사항
- 가변 길이 시퀀스에 대해 입력 패딩을 올바르게 처리해야 합니다.
- 입력과 출력 텐서를 올바르게 맞춰 배치 처리를 지원해야 합니다.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# Define a Transformer Model
#TODO: Implement a Transformer model
class TransformerModel(nn.Module):
    def __init__(self, input_dim, embed_dim, num_heads, num_layers, ff_dim, output_dim):
        super(TransformerModel, self).__init__()
        self.embedding = nn.Linear(input_dim, embed_dim)
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, activation='gelu', dim_feedforward=ff_dim, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer=self.encoder_layer, num_layers=num_layers) # 좀 마음에 안든다. RMSNorm
        self.ffn = nn.Linear(embed_dim, output_dim)
    def forward(self, x):
        # Define the forward pass logic
        x = self.embedding(x)
        x = self.encoder(x)
        # 마지막 토큰 선택하기(BERT)
        x = self.ffn(x[:,-1,:])
        return x

In [3]:
# Generate synthetic data
torch.manual_seed(42)
seq_length = 10
num_samples = 100
input_dim = 1
X = torch.rand(num_samples, seq_length, input_dim)  # Random sequences
y = torch.sum(X, dim=1)  # Target is the sum of each sequence

# Initialize the model, loss function, and optimizer
input_dim = 1
embed_dim = 16
num_heads = 2
num_layers = 2
ff_dim = 64
output_dim = 1

model = TransformerModel(input_dim, embed_dim, num_heads, num_layers, ff_dim, output_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

X

tensor([[[0.8823],
         [0.9150],
         [0.3829],
         [0.9593],
         [0.3904],
         [0.6009],
         [0.2566],
         [0.7936],
         [0.9408],
         [0.1332]],

        [[0.9346],
         [0.5936],
         [0.8694],
         [0.5677],
         [0.7411],
         [0.4294],
         [0.8854],
         [0.5739],
         [0.2666],
         [0.6274]],

        [[0.2696],
         [0.4414],
         [0.2969],
         [0.8317],
         [0.1053],
         [0.2695],
         [0.3588],
         [0.1994],
         [0.5472],
         [0.0062]],

        [[0.9516],
         [0.0753],
         [0.8860],
         [0.5832],
         [0.3376],
         [0.8090],
         [0.5779],
         [0.9040],
         [0.5547],
         [0.3423]],

        [[0.6343],
         [0.3644],
         [0.7104],
         [0.9464],
         [0.7890],
         [0.2814],
         [0.7886],
         [0.5895],
         [0.7539],
         [0.1952]],

        [[0.0050],
         [0.3068],
  

In [4]:
# Training loop
epochs = 1000
for epoch in range(epochs):
    # Forward pass
    predictions = model(X)
    loss = criterion(predictions, y)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Log progress every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [100/1000], Loss: 1.5725
Epoch [200/1000], Loss: 0.8917
Epoch [300/1000], Loss: 0.8080
Epoch [400/1000], Loss: 0.1878
Epoch [500/1000], Loss: 0.1064
Epoch [600/1000], Loss: 0.0631
Epoch [700/1000], Loss: 0.0519
Epoch [800/1000], Loss: 0.0445
Epoch [900/1000], Loss: 0.0352
Epoch [1000/1000], Loss: 0.0406


In [28]:
# Testing on new data
X_test = torch.rand(2, seq_length, input_dim)
with torch.no_grad():
    predictions = model(X_test)
    print(f"Predictions for {X_test.tolist()}: {predictions.tolist()}")

Predictions for [[[0.6648573279380798], [0.6041934490203857], [0.3187063932418823], [0.9813531041145325], [0.09837877750396729], [0.3223891258239746], [0.3124500513076782], [0.36122316122055054], [0.8705818057060242], [0.4751177430152893]], [[0.569571316242218], [0.05407053232192993], [0.16180634498596191], [0.8140731453895569], [0.34717607498168945], [0.6788632273674011], [0.11463749408721924], [0.21608346700668335], [0.7405895590782166], [0.8521053194999695]]]: [[5.090947151184082], [4.573826789855957]]
